In [6]:
import sys
import os
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))

from pyspark.sql import SparkSession
from src.config import SparkConfig

cfg = SparkConfig()
spark = (
    SparkSession.builder
    .appName(cfg.app_name)
    .master(cfg.master)
    .config("spark.sql.shuffle.partitions", str(cfg.shuffle_partitions))
    .config("spark.sql.adaptive.enabled", str(cfg.adaptive_enabled).lower())
    .config("spark.sql.adaptive.coalescePartitions.enabled",
            str(cfg.adaptive_enabled).lower())
    .config("spark.driver.memory", cfg.driver_memory)
    .config("spark.executor.memory", cfg.executor_memory)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [7]:
from src.data.schema import FEATURES_SCHEMA

In [8]:
path = Path().resolve().parent / "data" / "preprocessed" / "df_final_1.csv"

df = (
    spark.read
         .schema(FEATURES_SCHEMA)
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("nullValue", "")
         .option("mode", "DROPMALFORMED")
         .csv(str(path))
)

In [9]:
df.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- recency: integer (nullable = true)
 |-- frequency: integer (nullable = true)
 |-- monetary: double (nullable = true)
 |-- avg_basket: double (nullable = true)
 |-- n_products: integer (nullable = true)
 |-- n_countries: integer (nullable = true)
 |-- tenure_days: integer (nullable = true)
 |-- label: integer (nullable = true)



In [10]:
split = df.randomSplit([0.8, 0.2], seed=42)

In [11]:
from src.config import ModelConfig
from pyspark.ml.feature import VectorAssembler, StandardScaler

cfg = ModelConfig()
feature_cols = [
        "recency", "frequency", "monetary",
        "avg_basket", "n_products", "n_countries", "tenure_days",
    ]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip",
)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True, withStd=True,
)

In [12]:
from pyspark.ml.classification import (
    RandomForestClassifier, GBTClassifier, LogisticRegression
)

def classification(cfg : ModelConfig):
    if cfg.algorithm == "random_forest":
        return RandomForestClassifier(
            featuresCol="features",
            labelCol="label",
            numTrees=cfg.num_trees,
            maxDepth=cfg.max_depth,
            minInstancesPerNode=cfg.min_instances_per_node,
            seed=cfg.seed
        )
    
    elif cfg.algorithm == "gbt":
        return GBTClassifier(
            featuresCol="features",
            labelCol="label",
            maxIter=cfg.num_trees,
            maxDepth=cfg.max_depth,
            seed=cfg.seed
        )
    
    elif cfg.algorithm == "logistic":
        return LogisticRegression(
            featuresCol="features",
            labelCol="label",
            maxIter=100
        )
    
    else:
        raise ValueError(f"Algorithme inconnu : {cfg.algorithm}")

In [24]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [23]:
train, test = split

classifier = classification(cfg)
model = Pipeline(stages=[assembler, scaler, classifier])
model.fit(train)

PipelineModel_4ef00de50a9a

In [29]:
rf = model.getStages()[-1]
grid = (ParamGridBuilder().
        addGrid(rf.numTrees, [50, 100]).
        addGrid(rf.maxDepth, [6, 10]).
        build()
        )

evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
cv = CrossValidator(
    estimator=model,
    estimatorParamMaps=grid,
    evaluator=evaluator,
    numFolds=3,
    seed=cfg.seed,
    parallelism=2,
)

cv_model = cv.fit(train)
best_model = cv_model.bestModel

# Eval

In [30]:
preds = cv_model.transform(test)

In [32]:
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
)

auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(preds)

eval_mc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction"
)

metrics = {m:eval_mc.evaluate(preds,{eval_mc.metricName: m})
           for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]}
# or
# accuracy, f1, prec, rec = (eval_mc.evaluate(preds, {eval_mc.metricName: m})
#                            for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"])

cm = (preds.groupBy("label", "prediction")
            .count()
            .orderBy("label", "prediction")
            .collect())
cm_dict = {(int(r["label"]), int(r["prediction"])): r["count"] for r in cm}

In [37]:
classifier = best_model.stages[-1]
if hasattr(classifier, "featureImportances"):
    importances = list(zip(feature_cols, classifier.featureImportances.toArray()))
    importances.sort(key=lambda x: x[1], reverse=True)
    importances = [(f, round(float(i), 4)) for f, i in importances]
else:
    importances = []

In [39]:
result =  {
    "auc": round(auc, 4),
    "accuracy": round(metrics["accuracy"], 4),
    "f1": round(metrics["f1"], 4),
    "precision": round(metrics["weightedPrecision"], 4),
    "recall": round(metrics["weightedRecall"], 4),
    "confusion_matrix": cm_dict,
    "feature_importances": importances,
}

print(result)

{'auc': 0.7185, 'accuracy': 0.6561, 'f1': 0.6562, 'precision': 0.6568, 'recall': 0.6561, 'confusion_matrix': {(0, 0): 223, (0, 1): 120, (1, 0): 107, (1, 1): 210}, 'feature_importances': [('frequency', 0.3359), ('monetary', 0.2035), ('n_products', 0.1973), ('recency', 0.1239), ('tenure_days', 0.0757), ('avg_basket', 0.0637), ('n_countries', 0.0)]}
